# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Dataset description:", metadata.description)


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore metadata to find available record sets and fields
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- Record set @id: {rs['@id']} | name: {rs['name'] if 'name' in rs else 'N/A'}")
    print("  Fields (by @id):")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"    - {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
        else:
            print(f"    - {field}")
    print()


### Preview Records
Display a sample of records from a chosen record set, referencing by its `@id`.

In [ ]:
# Choose the primary record set @id
primary_record_set_id = None

# Find likely main record set (most fields, first in list)
if record_sets:
    primary_record_set_id = record_sets[0]['@id']

print(f"Showing sample records from record set @id: {primary_record_set_id}")
for idx, record in enumerate(dataset.records(record_set=primary_record_set_id)):
    print(record)
    if idx >= 2:
        break


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract all available record sets by their @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
    print(df.head(), "\n")

# Use the primary record set for further analysis
df_main = dataframes[primary_record_set_id]


## 4. Exploratory Data Analysis (EDA)
Process the data: filter records, normalize numeric fields, group by key attributes using their `@id`s.

In [ ]:
# Identify a numeric field for analysis by @id
# Let's search for a field whose dataType is 'Float' or 'Integer' in the main record set
numeric_field_id = None
group_field_id = None
main_record_set_fields = record_sets[0].get('field', [])
for field in main_record_set_fields:
    if isinstance(field, dict):
        if field.get('dataType') in ['Float', 'Integer'] and not numeric_field_id:
            numeric_field_id = field['@id']
        elif field.get('dataType') == 'Text' and not group_field_id:
            group_field_id = field['@id']

# If not found, fall back to any numeric column name
if not numeric_field_id and len(df_main.columns) > 0:
    # Try guessing with typical column names
    for col in df_main.columns:
        if 'score' in col or 'age' in col:
            numeric_field_id = col
            break

# Display selected field ids
print(f"Numeric field for analysis: {numeric_field_id}")
print(f"Group field for grouping: {group_field_id}")

# Filtering records where numeric field > threshold
threshold = 10
if numeric_field_id and numeric_field_id in df_main.columns:
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by group_field_id
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Visualize the numeric field distribution
if numeric_field_id and numeric_field_id in df_main.columns:
    plt.figure(figsize=(7,4))
    df_main[numeric_field_id].dropna().hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot grouped by group_field field
    if group_field_id and group_field_id in df_main.columns:
        plt.figure(figsize=(8,5))
        df_main.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich survey data on mental health indicators and demographics from Kilifi County, Kenya.
- Data extraction and processing is possible using the Croissant schema and `mlcroissant` library, referencing all entities by their `@id`.
- Numeric and categorical fields can be filtered, normalized, grouped, and visualized to explore trends and relationships in the data.
- This EDA can be extended to specific mental health scores (e.g., GAD-7, PHQ-9) and demographic analyses for further research.
